# Interactive 3D Fire-Fiscal Map (deck.gl)

Self-contained deck.gl page — Austin civic palette, four toggleable layers, rich click tooltips,
Carto basemap (no API key). Built parquet-free: the **response-area** and **landmark** layers render
now from data in the container; the **value-per-acre** (needs `parcels_value_per_acre_metro.parquet`)
and **stations** (needs `fire_stations.geojson`) layers activate automatically once those files land,
and the response-area layer recolors from a placeholder metric to true **net balance** once
`processed_data/fire_net_balance.geojson` exists (written by `notebooks/validation.ipynb`).

Output: `outputs/fire_fiscal_interactive_map.html`. Uses a hand-authored deck.gl template (not
pydeck's `to_html`) so the layer-toggle checkboxes are fully controllable.

In [1]:
import warnings; warnings.filterwarnings('ignore')
import json, sys
from pathlib import Path
import numpy as np
import geopandas as gpd

REPO = Path.cwd()
if REPO.name == 'notebooks': REPO = REPO.parent
PROC = REPO / 'processed_data'; OUT = REPO / 'outputs'; OUT.mkdir(exist_ok=True)
sys.path.insert(0, str(REPO))
import viz_palette as vp
SEQ = vp.seq_cmap(); DIV = vp.div_cmap()

ra = gpd.read_file(PROC/'response_areas_final.geojson').to_crs(4326)
# lighten geometry for the web (simplify in a projected CRS, back to lon/lat)
ra['geometry'] = ra.to_crs(2277).geometry.simplify(120).to_crs(4326)
ra = ra[ra.geometry.notna() & ~ra.geometry.is_empty].copy()
print(len(ra), 'response areas')


765 response areas


In [2]:
# ---- layer-2 colouring: true net balance if available, else a placeholder metric ----
NETBAL = PROC/'fire_net_balance.geojson'
if NETBAL.exists():
    nb_ = gpd.read_file(NETBAL)[['response_area_id','net_coverage_M']]
    ra = ra.merge(nb_, on='response_area_id', how='left')
    vmax = float(np.nanpercentile(np.abs(ra['net_coverage_M']), 98)) or 1.0
    ra['__net'] = ra['net_coverage_M']
    ra['__metric'] = None
    ra['__fill'] = ra['net_coverage_M'].apply(
        lambda v: vp.rgb_of_cont(DIV, v, -vmax, vmax, center=0, alpha=180)
        if v == v and v is not None else [200, 200, 200, 90])
    LAYER2_LABEL = 'Fire net balance ($M, coverage lens)'
    LAYER2_KIND = 'net'
else:
    METRIC = 'incidents_per_1000_units'   # present-data placeholder for the net-balance layer
    s = ra[METRIC].clip(lower=0)
    p98 = float(np.nanpercentile(s, 98)) or 1.0
    ra['__net'] = None
    ra['__metric'] = ra[METRIC].astype(float)
    ra['__fill'] = (s / p98).clip(0, 1).apply(lambda x: [*[int(round(c*255)) for c in SEQ(x)[:3]], 170])
    LAYER2_LABEL = 'Fire activity (incidents / 1k units) — placeholder for net balance'
    LAYER2_KIND = 'metric'
print('layer 2:', LAYER2_LABEL)

# trim properties to keep the embedded GeoJSON small
keep = ['response_area_id','RESPONSE_AREA_NAME','urban_class','__fill','__net','__metric']
ra_geo = ra[keep + ['geometry']].copy()
for c in ['__net','__metric']:
    ra_geo[c] = ra_geo[c].astype(object).where(ra_geo[c].notna(), None)
RA_JSON = json.loads(ra_geo.to_json())
clat = float(ra_geo.geometry.centroid.y.mean()); clon = float(ra_geo.geometry.centroid.x.mean())


layer 2: Fire net balance ($M, coverage lens)


In [3]:
# ---- colloquial landmark pins (hand-curated lat/lon; kind drives colour) ----
HI = [42,111,151]; LO = [156,66,33]; MIX = [233,196,106]
LANDMARKS = [
    {'name':'Downtown Austin','lon':-97.7431,'lat':30.2672,'color':HI,'note':'metro peak $/acre — high-rise office, condo, hotel'},
    {'name':'The Domain','lon':-97.7261,'lat':30.4007,'color':HI,'note':'dense north mixed-use — high value per acre'},
    {'name':'Mueller','lon':-97.7050,'lat':30.2980,'color':HI,'note':'redeveloped airport — dense, high productivity'},
    {'name':'West Lake Hills','lon':-97.8060,'lat':30.2930,'color':HI,'note':'low-density wealthy enclave — pays its way'},
    {'name':'Lakeway','lon':-97.9794,'lat':30.3637,'color':HI,'note':'big-lot high-value enclave — few road-miles'},
    {'name':'Bee Cave','lon':-97.9469,'lat':30.3088,'color':HI,'note':'high-value low-density — strengthens under road model'},
    {'name':'Northland / north-central','lon':-97.7340,'lat':30.3600,'color':MIX,'note':'mixed older north-central corridor'},
    {'name':'Round Rock','lon':-97.6789,'lat':30.5083,'color':MIX,'note':'growth suburb — old-town core is a local value peak'},
    {'name':'Georgetown','lon':-97.6770,'lat':30.6333,'color':LO,'note':'flips to net drain under road-cost model'},
    {'name':'Kyle','lon':-97.8770,'lat':29.9890,'color':LO,'note':'road-heavy growth suburb — net drain'},
    {'name':'Buda','lon':-97.8400,'lat':30.0855,'color':LO,'note':'road-heavy growth suburb — net drain'},
    {'name':'San Marcos','lon':-97.9414,'lat':29.8833,'color':LO,'note':'lots of pavement per dollar of value — net drain'},
]

# ---- staged layers (guarded) ----
HEX_JSON = None
PARQ = PROC/'parcels_value_per_acre_metro.parquet'
if PARQ.exists():
    import h3, pandas as pd
    g = gpd.read_parquet(PARQ)
    cen = g.to_crs(2277).geometry.centroid.to_crs(4326)
    hx = (pd.DataFrame({'hex':[h3.latlng_to_cell(y,x,8) for x,y in zip(cen.x,cen.y)],
                        'mv':g['market_value'].values,'ac':g['land_acres'].values})
          .groupby('hex').agg(mv=('mv','sum'),ac=('ac','sum')).reset_index())
    hx['vpa'] = hx['mv']/hx['ac']
    p99 = float(hx['vpa'].quantile(0.99)) or 1.0
    hx['elevation'] = (hx['vpa'].clip(upper=p99)/p99*6000).round()
    hx['color'] = (hx['vpa'].clip(upper=p99)/p99).apply(lambda x:[int(round(c*255)) for c in SEQ(x)[:3]]+[200])
    hx['vpa_label'] = hx['vpa'].apply(lambda v:f'${v:,.0f}/acre')
    HEX_JSON = hx[['hex','elevation','color','vpa_label']].to_dict('records')
    print(f'value/acre hexes: {len(HEX_JSON)}')
else:
    print('[staged] parquet absent — value/acre layer disabled')

STN_JSON = None
STN = PROC/'fire_stations.geojson'
if STN.exists():
    st = gpd.read_file(STN); st = st[st['DEPARTMENT']=='AFD']
    STN_JSON = [{'position':[float(g.x),float(g.y)],'station':str(r.get('STATION_NUMBER','')),
                 'resources':str(r.get('RESOURCES',''))} for r,g in zip(st.to_dict('records'), st.geometry)]
    print(f'stations: {len(STN_JSON)}')
else:
    print('[staged] fire_stations.geojson absent — stations layer disabled')


value/acre hexes: 8182
stations: 64


In [4]:
# ---- assemble the deck.gl HTML (placeholder-comment injection: no brace/$ collisions) ----
TEMPLATE = r'''<!DOCTYPE html>
<html><head><meta charset="utf-8"/><title>Austin Fire-Fiscal Interactive Map</title>
<script src="https://unpkg.com/deck.gl@9.0.0/dist.min.js"></script>
<script src="https://unpkg.com/maplibre-gl@3.6.2/dist/maplibre-gl.js"></script>
<link href="https://unpkg.com/maplibre-gl@3.6.2/dist/maplibre-gl.css" rel="stylesheet"/>
<style>
  html,body,#map{margin:0;width:100vw;height:100vh;overflow:hidden;font-family:-apple-system,Helvetica,Arial,sans-serif}
  #panel{position:absolute;top:12px;left:12px;z-index:10;background:rgba(251,250,246,.95);
    border:1px solid #d8d2c4;border-radius:8px;padding:12px 14px;max-width:300px;color:#1d3557;
    box-shadow:0 2px 10px rgba(0,0,0,.18);font-size:13px}
  #panel h3{margin:0 0 6px;font-size:14px;color:#1d3557}
  #panel label{display:block;margin:5px 0;cursor:pointer}
  #panel .note{color:#7a7a6a;font-size:11px;margin-left:22px}
  #panel input:disabled+span{color:#a8b0b8}
  .legend{margin-top:10px;border-top:1px solid #d8d2c4;padding-top:8px;font-size:11px}
  .bar{height:11px;border-radius:2px;margin:3px 0}
  .seq{background:linear-gradient(90deg,#1d3557,#457b9d,#a8dadc,#e9c46a,#bc6c25)}
  .div{background:linear-gradient(90deg,#9c4221,#cf8a5a,#efe9dd,#5a93b5,#2a6f97)}
  .row{display:flex;justify-content:space-between;color:#7a7a6a}
  .dot{display:inline-block;width:10px;height:10px;border-radius:50%;margin-right:5px;vertical-align:middle}
  #tip{font-size:12px;line-height:1.35}
</style></head>
<body>
<div id="map"></div>
<div id="panel">
  <h3>Austin Fire-Fiscal Map</h3>
  <label><input type="checkbox" id="t_vpa"/><span> Value per acre (3D)</span></label>
  <div class="note" id="n_vpa"></div>
  <label><input type="checkbox" id="t_net" checked/><span> /*LAYER2_LABEL*/</span></label>
  <label><input type="checkbox" id="t_stn"/><span> Fire stations</span></label>
  <div class="note" id="n_stn"></div>
  <label><input type="checkbox" id="t_lmk" checked/><span> Colloquial landmarks</span></label>
  <div class="legend">
    <div>value per acre (low &rarr; high)</div><div class="bar seq"></div>
    <div>net balance (drain &larr; 0 &rarr; contributor)</div><div class="bar div"></div>
    <div style="margin-top:6px"><span class="dot" style="background:#2a6f97"></span>high value / contributor
      &nbsp; <span class="dot" style="background:#9c4221"></span>net drain</div>
  </div>
</div>
<script>
const RA = /*RA*/;
const LANDMARKS = /*LANDMARKS*/;
const HEX = /*HEX*/;
const STATIONS = /*STATIONS*/;
const VIEW = /*VIEW*/;
const LAYER2_KIND = "/*LAYER2_KIND*/";
const METRIC_LABEL = "/*LAYER2_LABEL*/";

const state = {vpa:false, net:true, stn:false, lmk:true};

function tip(o){
  if(!o) return null;
  if(o.properties && o.properties.RESPONSE_AREA_NAME!==undefined){
    const p=o.properties; let s='<b>'+p.RESPONSE_AREA_NAME+'</b><br/>class: '+p.urban_class;
    if(p.__net!==undefined && p.__net!==null) s+='<br/>net coverage: $'+Number(p.__net).toFixed(1)+'M';
    else if(p.__metric!==undefined && p.__metric!==null) s+='<br/>incidents/1k units: '+Number(p.__metric).toFixed(1);
    return {html:s, style:{background:'#fbfaf6',color:'#1d3557',border:'1px solid #d8d2c4',
      borderRadius:'6px',padding:'6px 9px',fontSize:'12px'}};
  }
  if(o.name) return {html:'<b>'+o.name+'</b><br/>'+o.note, style:{background:'#1d3557',color:'#fff',
    borderRadius:'6px',padding:'6px 9px',fontSize:'12px',maxWidth:'220px'}};
  if(o.station) return {html:'<b>Station '+o.station+'</b><br/>'+o.resources, style:{background:'#1d3557',
    color:'#fff',borderRadius:'6px',padding:'6px 9px',fontSize:'12px'}};
  return null;
}

function buildLayers(){
  const L=[];
  if(HEX && state.vpa) L.push(new deck.H3HexagonLayer({id:'vpa', data:HEX, extruded:true,
    getHexagon:d=>d.hex, getElevation:d=>d.elevation, elevationScale:1, getFillColor:d=>d.color,
    pickable:true, opacity:0.85, coverage:0.95}));
  if(state.net) L.push(new deck.GeoJsonLayer({id:'net', data:RA, filled:true, stroked:true, extruded:false,
    getFillColor:f=>f.properties.__fill, getLineColor:[60,60,60,120], lineWidthMinPixels:0.5,
    pickable:true}));
  if(STATIONS && state.stn) L.push(new deck.ScatterplotLayer({id:'stn', data:STATIONS,
    getPosition:d=>d.position, getRadius:260, radiusMinPixels:4, radiusMaxPixels:14,
    getFillColor:[29,53,87], stroked:true, getLineColor:[255,255,255], lineWidthMinPixels:1, pickable:true}));
  if(state.lmk){
    L.push(new deck.ScatterplotLayer({id:'lmk', data:LANDMARKS, getPosition:d=>[d.lon,d.lat],
      getRadius:380, radiusMinPixels:6, radiusMaxPixels:18, getFillColor:d=>d.color,
      stroked:true, getLineColor:[255,255,255], lineWidthMinPixels:1.5, pickable:true}));
    L.push(new deck.TextLayer({id:'lmk-txt', data:LANDMARKS, getPosition:d=>[d.lon,d.lat],
      getText:d=>d.name, getSize:13, getColor:[20,25,45], getTextAnchor:'start',
      getAlignmentBaseline:'center', getPixelOffset:[10,0], fontWeight:600,
      outlineWidth:2, outlineColor:[255,255,255], fontSettings:{sdf:true}}));
  }
  return L;
}

const deckgl = new deck.DeckGL({
  container:'map',
  mapStyle:'https://basemaps.cartocdn.com/gl/positron-gl-style/style.json',
  initialViewState:VIEW, controller:true, getTooltip:({object})=>tip(object), layers:buildLayers()
});
function render(){ deckgl.setProps({layers:buildLayers()}); }

function wire(id,key,enabled,note){
  const el=document.getElementById(id);
  if(!enabled){ el.checked=false; el.disabled=true; state[key]=false; if(note) document.getElementById(note).textContent='(add data file to enable)'; }
  el.addEventListener('change',e=>{ state[key]=e.target.checked; render(); });
}
wire('t_vpa','vpa', HEX!==null, 'n_vpa');
wire('t_net','net', true, null);
wire('t_stn','stn', STATIONS!==null, 'n_stn');
wire('t_lmk','lmk', true, null);
</script></body></html>'''

import json as _json
html = (TEMPLATE
    .replace('/*RA*/', _json.dumps(RA_JSON))
    .replace('/*LANDMARKS*/', _json.dumps(LANDMARKS))
    .replace('/*HEX*/', _json.dumps(HEX_JSON) if HEX_JSON is not None else 'null')
    .replace('/*STATIONS*/', _json.dumps(STN_JSON) if STN_JSON is not None else 'null')
    .replace('/*VIEW*/', _json.dumps({'longitude':clon,'latitude':clat,'zoom':9.4,'pitch':45,'bearing':12}))
    .replace('/*LAYER2_KIND*/', LAYER2_KIND)
    .replace('/*LAYER2_LABEL*/', LAYER2_LABEL))

out_html = OUT/'fire_fiscal_interactive_map.html'
out_html.write_text(html)
print(f'wrote {out_html}  ({out_html.stat().st_size/1e6:.1f} MB)')


wrote /Users/chaseeasterling/GitHub/fire-incident-analysis/outputs/fire_fiscal_interactive_map.html  (1.9 MB)
